# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashmita-21/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My lane: Search performance and CTR opportunity scoring

**Task type: Scoring**

I want to identify content pages that receive fewer clicks than expected compared with pages appearing in a similar search-result position.

The system would calculate an opportunity score for each page and rank pages for human review. The content team could then investigate whether a page needs a better title, meta description, or closer alignment with search intent.

Scoring is useful because pages have different levels of visibility, search positions, and impression volumes. A ranked list can help the team decide which pages to investigate first.

The score is for decision support. A low score does not prove that a page is poor quality or that changing its title will improve performance.

In [1]:
# This cell is for CODE (numbers, a query, a check).

task_type = "Scoring"
decision = "Which content pages should the team review first?"
output = "A priority score for human review"

print("Task type:", task_type)
print("Decision:", decision)
print("Output:", output)

Task type: Scoring
Decision: Which content pages should the team review first?
Output: A priority score for human review


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target or proxy

My target is whether a page appears to have an opportunity to improve its click-through rate (CTR).

CTR is calculated as clicks divided by impressions. I can use historical impressions, clicks, and search position to understand past performance.

For an initial experiment, I could define a proxy label using a rule that identifies pages with lower-than-expected CTR compared with pages in similar search positions. This is a defined proxy, not a direct observation of whether a page will improve after an edit.

The label should be treated as a starting point for investigation, not proof that a page needs changing.

In [2]:
# This cell is for CODE (numbers, a query, a check).

target_type = "Defined proxy score"
proxy_name = "CTR gap relative to similar-position pages"
observed_future_outcome_available = False

print("Target type:", target_type)
print("Proxy:", proxy_name)
print("Observed post-edit outcomes available:",
      observed_future_outcome_available)

Target type: Defined proxy score
Proxy: CTR gap relative to similar-position pages
Observed post-edit outcomes available: False


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric

My primary evaluation metric will be Precision@20.

This measures how many of the first 20 pages selected by the model are relevant opportunities according to the defined proxy label.

A higher Precision@20 means the content team can review a more focused list of pages. I will compare the model against a simple baseline to see whether it provides value beyond a fixed rule.

The metric evaluates agreement with the proxy label; it does not prove that making changes to the selected pages will increase clicks.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
metric = "Precision@20"
required_evidence = "Human-reviewed opportunity labels"

print("Planned success metric:", metric)
print("Evidence required:", required_evidence)
print(
    "Can Precision@20 be measured before pages are reviewed?",
    False
)

Planned success metric: Precision@20
Evidence required: Human-reviewed opportunity labels
Can Precision@20 be measured before pages are reviewed? False


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of analysis

My unit of analysis is one content page for a defined observation period.

Each row should represent one page, with aggregated measures such as impressions, clicks, average search position, and CTR.

This structure lets me compare pages using consistent measurements. I will inspect the dataframe and its columns before deciding which features can be used safely for modelling.

In [4]:
# This cell is for CODE (numbers, a query, a check).

import pandas as pd
import numpy as np

# Load the public starter dataset from your GitHub repository.
data_url = (
    "https://raw.githubusercontent.com/"
    "rashmita-21/Flyrank-ML-Internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

# Check that the expected columns exist.
required_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing expected columns: {missing_columns}"
    )

# Keep pages with sufficient impressions and a valid position.
pages = df.loc[
    (df["impressions_90d"] >= 100)
    & (df["avg_position"] > 0)
].copy()

# Calculate the median CTR for each search-position tier.
pages["peer_median_ctr"] = (
    pages.groupby("position_tier")["ctr"]
    .transform("median")
)

# Positive values indicate CTR below the tier's median.
pages["ctr_gap"] = (
    pages["peer_median_ctr"] - pages["ctr"]
)

# Larger positive gaps receive higher priority scores.
pages["opportunity_score"] = pages["ctr_gap"].clip(lower=0)

# Show the unit of analysis and selected measurements.
display(
    pages[
        [
            "content_id",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "position_tier",
            "peer_median_ctr",
            "opportunity_score"
        ]
    ].head(10)
)

print("Original dataset rows:", len(df))
print("Pages retained after filtering:", len(pages))
print("Unique content items:", pages["content_id"].nunique())

,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier,peer_median_ctr,opportunity_score
0,content_304f48230142,3803,29,0.76,10.6,striking,0.15,0.00
1,content_a1fb4e703a9e,15320,7,0.05,20.3,page_3_5,0.06,0.01
2,content_9aa793d4d895,12581,11,0.09,36.5,page_3_5,0.06,0.00
3,content_331d6c4de07b,11751,58,0.49,6.2,page_1,0.23,0.00
4,content_d99b7a2d90ca,19140,24,0.13,44.0,page_3_5,0.06,0.00
5,content_d4084a4bc775,3970,1,0.03,8.5,page_1,0.23,0.20
7,content_a63219c6e95a,1724,1,0.06,21.2,page_3_5,0.06,0.00
8,content_5e6c160719bc,32574,29,0.09,46.0,page_3_5,0.06,0.00
9,content_c27558df2b0c,1240,2,0.16,4.9,page_1,0.23,0.07
10,content_d8ee6cc6d642,20919,324,1.55,2.2,top_3,0.19,0.00


Original dataset rows: 30000
Pages retained after filtering: 22006
Unique content items: 22006


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML may be useful

A fixed rule can identify pages with low CTR, but CTR varies with search position, impression volume, and the type of query a page appears for.

A model may combine several signals to help prioritise pages for review rather than relying on one threshold.

However, ML is not automatically better. I need to compare its results with a simple baseline and check whether it improves the quality of the selected pages. The output will support human decisions rather than automatically changing content.

In [5]:
# This cell is for CODE (numbers, a query, a check).

# Compare a simple overall-CTR rule with the
# position-adjusted opportunity score.

overall_median_ctr = pages["ctr"].median()

pages["simple_rule_flag"] = (
    pages["ctr"] < overall_median_ctr
)

# Top 20 pages by each approach.
simple_top20 = set(
    pages.nsmallest(20, "ctr")["content_id"]
)

scored_top20 = set(
    pages.nlargest(20, "opportunity_score")["content_id"]
)

overlap = simple_top20 & scored_top20

print("Overall median CTR:", round(overall_median_ctr, 3))
print("Pages selected by both approaches:", len(overlap), "out of 20")
print("Simple rule candidates:", len(simple_top20))
print("Position-adjusted candidates:", len(scored_top20))

display(
    pages.nlargest(10, "opportunity_score")[
        [
            "content_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "position_tier",
            "peer_median_ctr",
            "opportunity_score"
        ]
    ]
)

Overall median CTR: 0.14
Pages selected by both approaches: 4 out of 20
Simple rule candidates: 20
Position-adjusted candidates: 20


,content_id,impressions_90d,ctr,avg_position,position_tier,peer_median_ctr,opportunity_score
33,content_d87a116e2c79,298,0.0,6.8,page_1,0.23,0.23
66,content_d512773c9590,116,0.0,6.8,page_1,0.23,0.23
84,content_5458b7cf0d93,136,0.0,8.1,page_1,0.23,0.23
94,content_9983d31c53cb,7737,0.0,5.5,page_1,0.23,0.23
120,content_8f2559c3bc1b,740,0.0,5.7,page_1,0.23,0.23
145,content_65629be9dbb5,1230,0.0,7.0,page_1,0.23,0.23
151,content_d11eb2b0b223,141,0.0,5.7,page_1,0.23,0.23
166,content_96edab39e5fb,179,0.0,8.0,page_1,0.23,0.23
171,content_214a94adfa00,1168,0.0,6.9,page_1,0.23,0.23
183,content_ce611830d125,1409,0.0,6.9,page_1,0.23,0.23


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.